# DPE aggregation verification on Kaggle

Upload this file as a **new Kaggle notebook**, select a **GPU** (T4 is suitable), enable **Internet**, then choose **Run All**. Do not overwrite the original `noerula/dpe-reweighting` notebook.

This notebook keeps **Waterbirds, ResNet-50, and test worst-group accuracy (WGA)** fixed. Each of the 30 configurations is an independent experiment; no candidate-selection split or controlled background interventions are introduced.

**Default mode — `verify_existing`:** retrieve immutable version-4 evidence, check the source and input hashes, run software checks, freshly fit heads and all 30 aggregators, test ten combiner seeds for plain/dropout MLP, compute paired WGA intervals, and regenerate predictions from all test images using the original backbones on your Kaggle GPU. It does **not** retrain the original 100-epoch backbones. Allow roughly **10–30 minutes**, depending on downloads and runtime; this is an estimate, not a measured Kaggle duration for this notebook.

**Later mode — `additional_seeds`:** set `MODE` below and use `ADDITIONAL_SEEDS = [3]` to train one additional 100-epoch backbone/head seed with the same recipe. This is much heavier: the original run suggests roughly **1–2 hours per seed on T4**, with substantial runtime variation. Start with one seed per session; do not launch all five additional seeds accidentally. Backbone checkpoints are saved after every epoch and can resume from a preserved working directory.

The notebook fetches the pinned public implementation commit automatically. No GitHub push, Kaggle API key, attachment or private dataset is required for the default download path. If the original public notebook's latest version changes, it stops rather than substituting another experiment; attach the original version-4 output and set `ORIGINAL_OUTPUT_DIR` in that case.


In [ ]:
MODE = "verify_existing"              # Or "additional_seeds"; start with the default.
ORIGINAL_SEEDS = [0, 1, 2]
ADDITIONAL_SEEDS = [3]                 # Used only in additional_seeds mode.
COMBINER_SEEDS = list(range(10000, 10010))

RUN_REPOSITORY_TESTS = True
REPLAY_HEAD_TRAINING = True
RUN_COMBINER_STABILITY = True
RUN_TEST_BOOTSTRAP = True
RUN_IMAGE_REPLAY = True                # Requires GPU; false performs cached verification only.
BOOTSTRAP_REPLICATES = 10000
BOOTSTRAP_SEED = 20261112

WORK_DIR = "/kaggle/working/dpe_aggregation_verification"
ORIGINAL_OUTPUT_DIR = None            # Optional mounted ORIGINAL version-4 output directory.
DATA_DIR = None                       # Optional existing official Waterbirds directory.
RESUME_FROM = None                    # Optional previous full WORK_DIR mounted under /kaggle/input.
IMAGE_BATCH_SIZE = 256                # Reduce to 128 if GPU memory is insufficient.
WORKERS = 2
CPU_THREADS = 2


## Prepare the pinned source and isolated dependencies

Pillow **13.0.0** is intentional. The audit found that Pillow 10–12 can change resizing and the resulting features on some images, even when the JPEG bytes are identical. Pillow 13 reproduced the original cached features on the completed diagnostic and seed-0 image checks. The original run did not record its Pillow version, so this is an empirically verified compatibility choice, not a claim about its recorded environment.

Dependencies are installed into a worker-specific directory before the worker imports numerical/image libraries. This avoids stale Pillow imports in the notebook kernel. NumPy and SciPy match the original run. The supplied Kaggle PyTorch/CUDA installation is retained and its versions are recorded.

For a new session that needs to resume a backbone, add the previous notebook's **full output** as input and set `RESUME_FROM` to its saved working directory. The compact results ZIP excludes large backbones and raw features; it is not sufficient for backbone resume. Keep the same `WORK_DIR` and recipe. No original evidence is overwritten.


In [ ]:
import importlib.util
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

if MODE not in {"verify_existing", "additional_seeds"}:
    raise ValueError("Choose verify_existing or additional_seeds")
if not ORIGINAL_SEEDS or len(ORIGINAL_SEEDS) != len(set(ORIGINAL_SEEDS)) or any(s not in (0, 1, 2) for s in ORIGINAL_SEEDS):
    raise ValueError("Original seeds must be distinct values from 0, 1, 2")
if not ADDITIONAL_SEEDS or len(ADDITIONAL_SEEDS) != len(set(ADDITIONAL_SEEDS)) or any(type(s) is not int or s < 3 for s in ADDITIONAL_SEEDS):
    raise ValueError("Additional seeds must be distinct integers >=3")
if len(COMBINER_SEEDS) != len(set(COMBINER_SEEDS)) or not COMBINER_SEEDS:
    raise ValueError("Declare distinct combiner seeds")
if any(importlib.util.find_spec(m) is None for m in ("torch", "torchvision", "pandas")):
    raise RuntimeError("Use Kaggle's standard Python GPU environment with torch, torchvision and pandas.")

work = Path(WORK_DIR).expanduser().resolve()
if RESUME_FROM:
    prior = Path(RESUME_FROM).resolve()
    if not prior.is_dir():
        raise FileNotFoundError(prior)
    if work.exists() and any(work.iterdir()):
        raise RuntimeError("Restore only into a fresh WORK_DIR. For an already-restored session, set RESUME_FROM=None.")
    shutil.copytree(prior, work)
work.mkdir(parents=True, exist_ok=True)
repo = work / "reference_repo"
commit = "8e14249f4b3510363be06f627182e2daa9926179"
if not (repo / ".git").is_dir():
    repo.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "-q", str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "remote", "add", "origin", "https://github.com/chautanphat/waterbirds-shortcut-mitigation.git"], check=True)
try:
    head = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL).strip()
except subprocess.CalledProcessError:
    head = None
if head != commit:
    if subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain"], text=True).strip():
        raise RuntimeError("The private reference checkout was modified. Preserve it and choose a clean working directory.")
    subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", commit], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "FETCH_HEAD"], check=True)

runtime = work / "runtime_packages"
requirements = ["Pillow==13.0.0", "numpy==2.1.3", "scipy==1.16.3", "PyYAML>=6,<7", "pytest>=8,<10"]
marker = runtime / ".verification_requirements.json"
if not marker.is_file() or json.loads(marker.read_text()) != requirements:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--target", str(runtime), *requirements], check=True)
    marker.write_text(json.dumps(requirements))
config = {name: globals()[name] for name in [
    "MODE", "ORIGINAL_SEEDS", "ADDITIONAL_SEEDS", "COMBINER_SEEDS", "RUN_REPOSITORY_TESTS",
    "REPLAY_HEAD_TRAINING", "RUN_COMBINER_STABILITY", "RUN_TEST_BOOTSTRAP", "RUN_IMAGE_REPLAY",
    "BOOTSTRAP_REPLICATES", "BOOTSTRAP_SEED", "WORK_DIR", "ORIGINAL_OUTPUT_DIR", "DATA_DIR",
    "IMAGE_BATCH_SIZE", "WORKERS", "CPU_THREADS"]}
config["WORK_DIR"] = str(work)
config_path = work / "config.json"
config_path.write_text(json.dumps(config, indent=2))
print("Prepared source commit:", commit)
print("Mode:", MODE, "| Source seeds:", ORIGINAL_SEEDS if MODE == "verify_existing" else ADDITIONAL_SEEDS)
print("Working directory:", work)


## Verification implementation

The worker code below is embedded so this notebook can be uploaded alone. It checks immutable source/input identities, fits every declared configuration before test evaluation, and reports mismatches rather than silently accepting them. It uses an independent NumPy forward calculation and explicit group counts to check the reported WGA.

Additional-seed mode calls the preserved low-level backbone/head/combiner functions directly. This extends the source seed values beyond the old study wrapper's 0/1/2 allowlist without changing the model, data split, optimizer, epochs, covariance settings or aggregation grid. Each new source seed is reported separately.


In [ ]:
RUNNER_SOURCE = r'''"""Kaggle-side verification; this file is embedded in the uploadable notebook."""
import csv, hashlib, json, os, platform, shutil, subprocess, sys, time, urllib.parse, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

C = json.loads(Path(sys.argv[1]).read_text())
W = Path(C['WORK_DIR']); REPO = W / 'reference_repo'
RUN_TAG = C['MODE'] + '-' + hashlib.sha256(json.dumps(C, sort_keys=True).encode()).hexdigest()[:10]
OUT = W / 'results' / RUN_TAG
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
import numpy as np
import scipy
import torch
import PIL
from src.methods.dpe_head_reweighting import aggregation_source as sources
from src.methods.dpe_head_reweighting import ensemble, neural_combiner as neural, weighting_rules as rules
from src.methods.dpe_head_reweighting import stage0
from src.methods.dpe_head_reweighting.rule_suite import notebook_code_hash
from src.datasets.waterbirds import Waterbirds
from src.models.resnet import load_resnet_checkpoint
from torch.utils.data import DataLoader

COMMIT = '8e14249f4b3510363be06f627182e2daa9926179'
SCRIPT_VERSION = '367976457'
STUDY_ID = 'aggregation-standard-seeds012-0dcf563966c5'
GROUPS = ['00', '01', '10', '11']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(C['CPU_THREADS'])


def log(message):
    print(time.strftime('%H:%M:%S'), message, flush=True)


def write(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + '\n')
    temporary.replace(path)


def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()


def table(path, rows):
    if not rows: return
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)


def archive():
    """Compact reports/models, not images, large features, or backbone checkpoints."""
    target = W / 'verification_results.zip'; temporary = W / 'verification_results.zip.tmp'
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as z:
        for root in [W / 'results', W / 'logs']:
            if root.exists():
                for p in root.rglob('*'):
                    if p.is_file() and p.suffix not in ['.pt', '.npy']:
                        z.write(p, p.relative_to(W).as_posix())
        z.write(W / 'config.json', 'config.json')
    temporary.replace(target)
    return target


def download(url, path):
    path = Path(path)
    if path.is_file(): return
    path.parent.mkdir(parents=True, exist_ok=True); part = path.with_suffix(path.suffix + '.part')
    with urllib.request.urlopen(url, timeout=120) as response, part.open('wb') as stream:
        shutil.copyfileobj(response, stream, length=1024 * 1024)
    part.replace(path)


def get_reference():
    evidence = W / 'original_evidence' / 'dpe_head_aggregation_session'
    study = evidence / 'studies' / STUDY_ID
    # A mounted notebook output may contain version 4's complete directory.
    mounted = C.get('ORIGINAL_OUTPUT_DIR')
    if not mounted:
        for root in Path('/kaggle/input').rglob(STUDY_ID):
            if root.is_dir() and (root / 'plan.json').is_file():
                mounted = str(root.parent.parent); break
    if mounted:
        mounted = Path(mounted)
        if mounted.name != 'dpe_head_aggregation_session':
            mounted = mounted / 'dpe_head_aggregation_session'
        if not (mounted / 'studies' / STUDY_ID / 'plan.json').is_file():
            raise ValueError('ORIGINAL_OUTPUT_DIR must contain the version-4 study directory.')
    index, token, page = [], None, 0
    if not mounted:
        # Public endpoint: never require or print Kaggle credentials or signed URLs.
        while True:
            args = {'userName': 'noerula', 'kernelSlug': 'dpe-reweighting', 'pageSize': 500}
            if token: args['pageToken'] = token
            url = 'https://www.kaggle.com/api/v1/kernels/output?' + urllib.parse.urlencode(args)
            with urllib.request.urlopen(url, timeout=60) as response: body = json.load(response)
            files = body.get('files', [])
            if any('/kf/' + SCRIPT_VERSION + '/' not in item['url'] for item in files):
                raise RuntimeError('The public notebook now points to another version. Add the ORIGINAL version-4 notebook output as Kaggle Input and set ORIGINAL_OUTPUT_DIR. Never substitute a later version.')
            index.extend(files); page += 1
            if page % 5 == 0: log(f'Listed {page} reference-output pages')
            token = body.get('nextPageToken')
            if not token: break
        lookup = {item['fileName']: item['url'] for item in index}
    else:
        lookup = {'dpe_head_aggregation_session/' + p.relative_to(mounted).as_posix(): p
                  for p in mounted.rglob('*') if p.is_file()}
    wanted = []
    for name, location in lookup.items():
        if '/repository-' in name: continue
        take = ('/studies/' + STUDY_ID + '/' in name and name.endswith(('/plan.json', '/models.json', '/metrics.json', '/summary.json')))
        take |= ('/head_sources/' in name and name.endswith(('/ensemble.pt', '/manifest.json', '/heads.json', '/combiner_training.npz', '/test.npz')))
        take |= ('/features/' in name and name.endswith(('/metadata.csv', '/handoff.json', '/provenance.json')))
        if C['RUN_IMAGE_REPLAY'] or C['REPLAY_HEAD_TRAINING']:
            take |= ('/features/' in name and name.endswith(('/val_raw.npz', '/test_raw.npz')))
        if C['RUN_IMAGE_REPLAY']:
            take |= ('/backbones/' in name and name.endswith('/ckpt_last.pt'))
        if take: wanted.append((name, location))
    def fetch(item):
        name, location = item; target = evidence.parent / name
        if mounted:
            if not target.exists(): target.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(location, target)
        else: download(location, target)
        return name
    for _ in ThreadPoolExecutor(max_workers=4).map(fetch, wanted): pass
    if not (study / 'plan.json').is_file(): raise RuntimeError('Missing immutable version-4 plan.json.')
    plan = json.loads((study / 'plan.json').read_text())
    for name, expected in plan['implementation'].items():
        actual = notebook_code_hash(REPO / name) if name.endswith('.ipynb') else sha(REPO / name)
        if actual != expected: raise RuntimeError('Pinned implementation hash mismatch: ' + name)
    write(OUT / 'reference_plan.json', plan)
    write(OUT / 'reference_identity.json', dict(kaggle_version=4, script_version=SCRIPT_VERSION, study_id=STUDY_ID, source_commit=COMMIT))
    return evidence, study, plan


def source_data(source):
    manifest = json.loads((source / 'experiment/manifest.json').read_text())
    heads = json.loads((source / 'experiment/heads.json').read_text())
    if sha(source / 'checkpoints/ensemble.pt') != heads['lock']['ensemble']: raise ValueError('Head hash mismatch')
    if sha(source / 'experiment/manifest.json') != heads['lock']['manifest']: raise ValueError('Manifest hash mismatch')
    if manifest['parts'] != sources.partition(manifest['validation']): raise ValueError('Partition mismatch')
    parts = manifest['parts']; a = {r['id'] for r in parts['head_training']}; b = {r['id'] for r in parts['combiner_training']}
    if len(a) != 720 or len(b) != 479 or a & b: raise ValueError('Invalid 720/479 separation')
    p, y, g, ids = sources.load_cache(source, 'combiner_training', manifest, heads['head_ids'], heads['lock']['ensemble'])
    return manifest, heads, (p, y, g, ids)


def metrics(p, y, groups):
    # Independent count calculation, including argmax ties going to class 0.
    correct = np.asarray(p).argmax(-1) == y
    counts = {g: int(np.sum(groups == g)) for g in GROUPS}
    if min(counts.values()) == 0: raise ValueError('Missing group')
    accuracy = {g: float(correct[groups == g].mean()) for g in GROUPS}
    return {'wga': min(accuracy.values()), 'overall': float(correct.mean()), 'groups': accuracy, 'counts': counts}


def numpy_predict(row, p):
    """Independent forward calculation: does not call the model's PyTorch forward."""
    p = np.asarray(p, dtype=np.float64)
    if row['family'] == 'uniform': return np.einsum('h,hnc->nc', np.ones(len(p)) / len(p), p)
    if 'weights' in row: return np.einsum('h,hnc->nc', np.asarray(row['weights']), p)
    model = row['model']; state = {k: np.asarray(v) for k, v in model['state'].items()}
    q = p[:, :, 1].T; x = 2 * q - 1
    if model['kind'] == 'logistic': z = x @ state['network.weight'].T + state['network.bias']
    else: z = np.tanh(x @ state['network.0.weight'].T + state['network.0.bias']) @ state['network.2.weight'].T + state['network.2.bias']
    if model['kind'] == 'gate':
        weight = np.exp(z - z.max(1, keepdims=True)); weight /= weight.sum(1, keepdims=True)
        value = (weight * q).sum(1)
    else: value = 1 / (1 + np.exp(-z[:, 0]))
    return np.column_stack([1 - value, value])


def fit_configurations(p, y, g, config, source_seed):
    # Same declared 30 procedures; direct fit supports new source seeds without
    # changing the preserved library's legacy 0/1/2 study-wrapper restriction.
    rows = [dict(id='uniform', family='uniform', parameters={}, model=None, fit=None)]
    for family in neural.FAMILIES:
        dropout = config['prediction_dropout'] if family.endswith('_dropout') else 0.
        for wd in config['weight_decays']:
            model, fitted = neural.fit(p, y, g, config, family.split('_')[0], wd, dropout,
                                       source_seed + config['seed_offset'])
            rows.append(dict(id=f'{family}-wd{wd}', family=family,
                             parameters=dict(weight_decay=wd, prediction_dropout=dropout), model=model, fit=fitted))
    rows += [r for r in rules.fit_all_rules(p, y, g, config['weighting_rules']) if r['id'] != 'uniform']
    if len(rows) != 30 or len({r['id'] for r in rows}) != 30: raise ValueError('Incomplete configuration grid')
    return rows


def get_dataset():
    explicit = C.get('DATA_DIR')
    if explicit:
        if not Path(explicit).is_dir(): raise FileNotFoundError(explicit)
        return Path(explicit)
    # Official script downloads once, verifies official counts, and prepares metadata.
    target = W / 'data'
    if not (target / 'waterbirds/metadata_waterbirds.csv').is_file():
        subprocess.run([sys.executable, 'scripts/prepare_waterbirds.py', '--data_dir', str(target)], check=True)
    return target / 'waterbirds'


def image_replay(seed, source, checkpoint, feature_directory, dataset, frozen_rows, reference_p, y, groups):
    if DEVICE != 'cuda': raise RuntimeError('Enable a Kaggle GPU for image replay, or set RUN_IMAGE_REPLAY=False for cached verification only.')
    manifest = json.loads((source / 'experiment/manifest.json').read_text())
    if sha(checkpoint) != manifest['backbone_hash']: raise ValueError('Backbone checkpoint hash differs from source manifest')
    handoff = json.loads((feature_directory / 'handoff.json').read_text())
    if sha(feature_directory / 'test_raw.npz') != handoff['artifacts']['test_raw.npz']: raise ValueError('Raw test feature hash mismatch')
    head_record = ensemble.load(source / 'checkpoints/ensemble.pt', 2048, 15)
    model = load_resnet_checkpoint(checkpoint, device=DEVICE); model.eval()
    ds = Waterbirds(dataset, 'test')
    with np.load(feature_directory / 'test_raw.npz', allow_pickle=False) as cache:
        old_raw = cache['features']; expected_ids = cache['ids'].astype(str)
    if ds.filenames != expected_ids.tolist(): raise ValueError('Image/feature ID order differs')
    loader = DataLoader(ds, batch_size=C['IMAGE_BATCH_SIZE'], shuffle=False, num_workers=C['WORKERS'])
    chunks = []
    with torch.inference_mode():
        for index, (_, inputs, _, _) in enumerate(loader):
            chunks.append(model.extract_features(inputs.to(DEVICE)).cpu().numpy())
            if index % 10 == 0: log(f'GPU image replay seed {seed}: {sum(len(x) for x in chunks)}/{len(ds)}')
    raw = np.concatenate(chunks)
    p = ensemble.predict(head_record, raw, batch_size=C['IMAGE_BATCH_SIZE'], device=DEVICE)
    report = dict(source_seed=seed,images=len(ds),pillow=PIL.__version__,
                  feature_max_abs_diff=float(np.abs(raw-old_raw).max()),
                  feature_relative_l2=float(np.linalg.norm(raw-old_raw)/np.linalg.norm(old_raw)),
                  head_probability_max_abs_diff=float(np.abs(p-reference_p).max()),
                  changed_head_decisions=int((p.argmax(2)!=reference_p.argmax(2)).sum()), configurations={})
    for row in frozen_rows:
        old=numpy_predict(row,reference_p); fresh=numpy_predict(row,p)
        report['configurations'][row['id']]=dict(metrics=metrics(fresh,y,groups),
                changed_decisions=int((old.argmax(1)!=fresh.argmax(1)).sum()),
                probability_max_abs_diff=float(np.abs(old-fresh).max()))
    write(OUT / f'images/seed{seed}.json', report)
    del model; torch.cuda.empty_cache()
    log(f'Finished GPU image replay seed {seed}; results recorded for all 30 configurations')
    return report


def bootstrap(seed_records, names):
    if not seed_records: return
    correctness=[]; expected=None
    for rec in seed_records:
        p,y,g,ids=rec['test']; rows={r['id']:r for r in rec['original_rows']}
        if expected is not None and not np.array_equal(ids,expected): raise ValueError('Cross-seed test IDs differ')
        expected=ids
        correctness.append([numpy_predict(rows[name],p).argmax(1)==y for name in names])
    a=np.asarray(correctness); ns=len(a); rng=np.random.default_rng(C['BOOTSTRAP_SEED']); draws=[]
    for group in GROUPS:
        mask=g==group; patterns,counts=np.unique(a[:,:,mask].reshape(ns*len(names),-1).T,axis=0,return_counts=True)
        sampled=rng.multinomial(int(mask.sum()),counts/counts.sum(),size=C['BOOTSTRAP_REPLICATES'])
        draws.append((sampled@patterns.astype(float)/mask.sum()).reshape(C['BOOTSTRAP_REPLICATES'],ns,len(names)))
    b=np.min(draws,axis=0); point=np.min([a[:,:,g==group].mean(2) for group in GROUPS],axis=0); results=[]
    for wd in [.01,.001]:
        method=f'mlp_dropout-wd{wd}'
        for reference in ['uniform',f'mlp-wd{wd}']:
            i,j=names.index(method),names.index(reference); delta=100*(b[:,:,i]-b[:,:,j]); observed=100*(point[:,i]-point[:,j])
            lo,hi=np.percentile(delta.mean(1),[2.5,97.5]); results.append(dict(method=method,reference=reference,
                mean_delta_pp=float(observed.mean()),per_seed_delta_pp=observed.tolist(),conditional_95_percentile_pp=[float(lo),float(hi)]))
    write(OUT/'paired_bootstrap.json',dict(replicates=C['BOOTSTRAP_REPLICATES'],seed=C['BOOTSTRAP_SEED'],
        interpretation='Pointwise paired group-stratified test-example intervals conditional on these fitted models; not uncertainty across newly trained backbones. Same resamples across source seeds; minimum group recomputed each replicate.',results=results))


def contracts():
    # Independent, small numerical contracts. No test-label fitting.
    q=torch.tensor([.1,.2,.4,.6,.7,.8,.9,.6,.8,.7],dtype=torch.float64,requires_grad=True)
    y=torch.tensor([0.,0.,0.,1.,1.,1.,1.,1.,1.,1.],dtype=torch.float64)
    g=torch.tensor([0,1,1,2,2,2,3,3,3,3]); raw=q.detach().numpy(); labels=y.numpy()
    loss=-labels*np.log(raw)-(1-labels)*np.log(1-raw)
    expected=np.mean([loss[g.numpy()==i].mean() for i in range(4)])
    np.testing.assert_allclose(float(neural.group_loss(q,y,g)),expected,atol=1e-14)
    keep=neural.prediction_mask(torch.ones(100,15),1.,torch.Generator().manual_seed(0))
    assert bool((keep.sum(1)==1).all())
    with neural.cpu_randomness(0): model=neural.Combiner('mlp',15,8)
    inputs=torch.full((2,15),.8,dtype=torch.float64); mask=torch.ones_like(inputs,dtype=torch.bool);mask[:,0]=False
    captured=[];hook=model.network[0].register_forward_pre_hook(lambda m,args:captured.append(args[0].detach().numpy()))
    model(inputs,mask);hook.remove();np.testing.assert_allclose(captured[0][:,0],0);np.testing.assert_allclose(captured[0][:,1:],.6)
    write(OUT/'numerical_contracts.json',dict(group_balanced_loss='passed',all_dropped_fallback='passed',neutral_prediction_masking='passed'))


def run():
    if subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()!=COMMIT: raise RuntimeError('Wrong source commit')
    if PIL.__version__!='13.0.0': raise RuntimeError('Worker must load Pillow13.0.0; use the provided bootstrap cell.')
    environment=dict(python=platform.python_version(),torch=torch.__version__,torchvision=__import__('torchvision').__version__,
        numpy=np.__version__,scipy=scipy.__version__,pillow=PIL.__version__,cuda=torch.version.cuda,device=DEVICE,
        gpu=torch.cuda.get_device_name(0) if DEVICE=='cuda' else None,commit=COMMIT)
    write(OUT/'environment.json',environment);log('Environment: '+json.dumps(environment))
    if (C['RUN_IMAGE_REPLAY'] or C['MODE']=='additional_seeds') and DEVICE!='cuda': raise RuntimeError('Select GPU in Kaggle Settings before running.')
    contracts()
    if C['RUN_REPOSITORY_TESTS']:
        result=subprocess.run([sys.executable,'-m','pytest','tests/methods/dpe_head_reweighting','-q','-k','not notebook','--disable-warnings'],capture_output=True,text=True,
                             env={**os.environ,'PYTEST_DISABLE_PLUGIN_AUTOLOAD':'1'})
        (OUT/'repository_tests.txt').write_text(result.stdout+'\n'+result.stderr)
        if result.returncode: raise RuntimeError('Repository tests failed; inspect repository_tests.txt. Do not accept a benchmark result silently.')
    reference_plan=None; records=[]
    if C['MODE']=='verify_existing':
        evidence,study,reference_plan=get_reference(); config=reference_plan['config']
        for seed in C['ORIGINAL_SEEDS']:
            choices=list((evidence/'head_sources').glob(f'standard-seed{seed}-*'))
            if len(choices)!=1: raise ValueError('Expected exactly one original source for seed '+str(seed))
            source=choices[0];manifest,heads,train=source_data(source)
            original=json.loads((study/f'comparisons/seed{seed}/models.json').read_text())
            write(OUT/f'original_models/seed{seed}.json',original)
            for name,expected in original['lock']['inputs'].items():
                if sha(source/name)!=expected: raise ValueError('Original input hash mismatch: '+name)
            log(f'Freshly fitting all 30 configurations: original source seed {seed}')
            p,y,g,ids=train;fresh=fit_configurations(p,y,g,config,seed)
            write(OUT/f'frozen/seed{seed}.json',dict(source_seed=seed,config=config,candidates=fresh))
            feature=next((evidence/'features/standard').glob(f'seed{seed}-*')) if (C['RUN_IMAGE_REPLAY'] or C['REPLAY_HEAD_TRAINING']) else None
            if C['REPLAY_HEAD_TRAINING']:
                from src.methods.dpe_head_reweighting import train_heads
                with np.load(feature/'val_raw.npz') as cache: raw=cache['features']; val_ids=cache['ids'].astype(str)
                lookup={name:i for i,name in enumerate(val_ids)}; pool=manifest['parts']['head_training']; ix=[lookup[r['id']] for r in pool]
                new=train_heads.train(raw[ix],np.array([r['y'] for r in pool]),np.array([r['group'] for r in pool]),manifest['training'],seed,dict(verification='fresh head fit'),device=DEVICE)
                old=ensemble.load(source/'checkpoints/ensemble.pt'); ensemble.save(OUT/f'fresh_heads/seed{seed}.pt',new)
                write(OUT/f'head_replay/seed{seed}.json',dict(selected_epochs=new['selected_epochs'],original_selected_epochs=old['selected_epochs'],
                    prototype_max_abs_diff=float((new['prototypes']-old['prototypes']).abs().max()),distance_scale_max_abs_diff=float((new['distance_scales']-old['distance_scales']).abs().max())))
            records.append(dict(seed=seed,source=source,manifest=manifest,heads=heads,train=train,original_rows=original['candidates'],fresh=fresh,feature=feature,
                                checkpoint=evidence/f'backbones/standard/seed{seed}/ckpt_last.pt'))
    elif C['MODE']=='additional_seeds':
        import yaml
        config=yaml.safe_load((REPO/'configs/dpe_head_reweighting/aggregation_comparison.yaml').read_text());dataset=get_dataset()
        for seed in C['ADDITIONAL_SEEDS']:
            if type(seed)!=int or seed<3: raise ValueError('Additional seeds must be distinct new integers >=3.')
            log(f'Preparing NEW Waterbirds/ResNet50 seed {seed}; this trains a 100-epoch backbone and can take hours.')
            source,handoff=sources.build_source(W/'additional_sources',dataset,'standard',seed,DEVICE,C['WORKERS'],True)
            manifest,heads,train=source_data(source);p,y,g,ids=train
            fresh=fit_configurations(p,y,g,config,seed);write(OUT/f'frozen/seed{seed}.json',dict(source_seed=seed,config=config,candidates=fresh))
            # These new seeds use the same functions/recipe. We bypass only the
            # original study wrapper's 0/1/2 allowlist, not its scientific settings.
            records.append(dict(seed=seed,source=source,manifest=manifest,heads=heads,train=train,original_rows=fresh,fresh=fresh,
                feature=Path(handoff['test_features']).parent,checkpoint=Path(handoff['backbone']),handoff=handoff))
    else: raise ValueError('MODE must be verify_existing or additional_seeds')
    # Freeze every declared combiner-randomness fit before loading test caches.
    sensitivity=[]
    if C['RUN_COMBINER_STABILITY']:
        for rec in records:
            p,y,g,ids=rec['train']
            for random_seed in C['COMBINER_SEEDS']:
                for wd in config['weight_decays']:
                    for family,drop in [('mlp',0.),('mlp_dropout',config['prediction_dropout'])]:
                        model,fit=neural.fit(p,y,g,config,'mlp',wd,drop,random_seed)
                        sensitivity.append(dict(source_seed=rec['seed'],combiner_seed=random_seed,weight_decay=wd,family=family,model=model,fit=fit))
            log(f'Frozen combiner-randomness fits for source {rec["seed"]}')
        write(OUT/'stability/frozen_models.json',sensitivity)
    archive();log('All requested fits frozen; beginning test evaluation.')
    comparison=[];randomness=[]
    dataset=get_dataset() if C['RUN_IMAGE_REPLAY'] else None
    for rec in records:
        seed,source=rec['seed'],rec['source']
        if C['MODE']=='additional_seeds': sources.ensure_test_cache(rec['handoff'])
        if C['MODE']=='verify_existing':
            reference_report=json.loads((study/f'comparisons/seed{seed}/metrics.json').read_text())
            if sha(source/'experiment/predictions/test.npz') != reference_report['test_cache_hash']: raise ValueError('Original test cache hash mismatch')
            if sha(study/f'comparisons/seed{seed}/models.json') != reference_report['models_hash']: raise ValueError('Original model record hash mismatch')
        p,y,g,ids=sources.load_cache(source,'test',rec['manifest'],rec['heads']['head_ids'],rec['heads']['lock']['ensemble'])
        if len(y)!=5794 or [int(np.sum(g==group)) for group in GROUPS]!=[2255,2255,642,642]: raise ValueError('Wrong official test counts')
        rec['test']=(p,y,g,ids); original_rows={r['id']:r for r in rec['original_rows']};u=metrics(numpy_predict(original_rows['uniform'],p),y,g)
        for row in rec['fresh']:
            fresh,_=neural.candidate_prediction(row,p); independent=numpy_predict(row,p)
            np.testing.assert_allclose(fresh,independent,atol=1e-12,rtol=0)
            old=numpy_predict(original_rows[row['id']],p);m=metrics(fresh,y,g)
            if C['MODE']=='verify_existing':
                expected=reference_report['results'][row['id']]['metrics']
                old_metric=metrics(old,y,g)
                if abs(old_metric['wga']-expected['wga'])>1e-14 or old_metric['counts']!=expected['counts']: raise ValueError('Stored reference metrics do not reproduce')
            comparison.append(dict(source_seed=seed,configuration=row['id'],wga=m['wga'],delta_uniform_pp=100*(m['wga']-u['wga']),
                changed_reference_decisions=int((old.argmax(1)!=fresh.argmax(1)).sum()) if C['MODE']=='verify_existing' else None,probability_max_abs_diff=float(np.abs(old-fresh).max()),
                **{f'group_{group}':m['groups'][group] for group in GROUPS},
                **{f'count_{group}':m['counts'][group] for group in GROUPS},
                **{f'correct_{group}':int(round(m['groups'][group]*m['counts'][group])) for group in GROUPS}))
        for row in [r for r in sensitivity if r['source_seed']==seed]:
            prediction,_=neural.predict(row['model'],p);m=metrics(prediction,y,g)
            randomness.append({k:row[k] for k in ['source_seed','combiner_seed','weight_decay','family']}|
                              dict(wga=m['wga'],delta_uniform_pp=100*(m['wga']-u['wga'])))
        table(OUT/'per_seed.csv',comparison);table(OUT/'combiner_randomness.csv',randomness)
        if C['RUN_IMAGE_REPLAY']:
            image_replay(seed,source,rec['checkpoint'],rec['feature'],dataset,rec['original_rows'],p,y,g)
        archive();log(f'Seed {seed} verified and archived')
    names=['uniform','mlp-wd0.01','mlp-wd0.001','mlp_dropout-wd0.01','mlp_dropout-wd0.001']
    if C['RUN_TEST_BOOTSTRAP']: bootstrap(records,names)
    summary=[]
    for name in sorted({row['configuration'] for row in comparison}):
        rows=[r for r in comparison if r['configuration']==name];values=[r['wga'] for r in rows];delta=[r['delta_uniform_pp'] for r in rows]
        summary.append(dict(configuration=name,source_seed_count=len(rows),mean_wga=float(np.mean(values)),sample_sd_wga=float(np.std(values,ddof=1)) if len(values)>1 else None,
            mean_delta_uniform_pp=float(np.mean(delta)),sample_sd_delta_pp=float(np.std(delta,ddof=1)) if len(delta)>1 else None,
            changed_reference_decisions=sum(r['changed_reference_decisions'] for r in rows) if C['MODE']=='verify_existing' else None))
    table(OUT/'summary.csv',summary)
    lines=['# DPE aggregation verification results','','All procedures are independently reported. Accuracy values in CSV are fractions; deltas are percentage points.',
           'Mean and sample SD across source seeds are distinct from combiner randomness on fixed heads. No candidate-selection split is introduced.',
           'This is a cached/frozen-backbone verification unless MODE=additional_seeds. Original data and source are preserved.', '',
           '| Configuration | Mean test WGA (%) | Change vs matched uniform (pp) | Changed reference decisions |','| --- | ---: | ---: | ---: |']
    for r in summary:
        changed=r['changed_reference_decisions'] if r['changed_reference_decisions'] is not None else 'not applicable (new seed)'
        lines.append(f"| {r['configuration']} | {100*r['mean_wga']:.4f} | {r['mean_delta_uniform_pp']:+.4f} | {changed} |")
    lines+=['','Image replay details: `images/`; fresh head fitting: `head_replay/`; conditional test-example intervals: `paired_bootstrap.json`.',
            'Any mismatch is reported rather than hidden. A nonzero decision mismatch needs investigation; numerical closeness alone is not an exact replication.',
            'An observed gain over uniform does not establish that dropout beats plain MLP. Original and new source seeds should remain separately identifiable.']
    (OUT/'report.md').write_text('\n'.join(lines)+'\n');write(OUT/'status.json',dict(status='complete',mode=C['MODE'],source_seeds=[r['seed'] for r in records]))
    log('Completed. Download '+str(archive()))


try:
    run()
except BaseException as exc:
    write(OUT/'status.json',dict(status='interrupted' if isinstance(exc,KeyboardInterrupt) else 'failed',exception=type(exc).__name__,message=str(exc)))
    archive()
    raise
'''
runner_path = work / 'verification_runner.py'
runner_path.write_text(RUNNER_SOURCE)
print('Worker written:', runner_path)


## Run on your Kaggle resources

The log is streamed below and saved under `WORK_DIR/logs/`. A compact ZIP is refreshed after each completed seed and on a handled failure or interruption. If a check fails, retain its output rather than changing settings to improve the test score.

A GPU runtime consumes your Kaggle GPU allowance while it is enabled. All experiment computation in this cell runs in your Kaggle session, not in the Codex cloud environment.


In [ ]:
import signal

logs = work / "logs"
logs.mkdir(exist_ok=True)
env = os.environ.copy()
env["PYTHONPATH"] = str(runtime) + os.pathsep + str(repo) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
env["OMP_NUM_THREADS"] = str(CPU_THREADS)
env["MKL_NUM_THREADS"] = str(CPU_THREADS)
env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
log_path = logs / (MODE + ".log")
with log_path.open("w", buffering=1) as logfile:
    process = subprocess.Popen([sys.executable, "-u", str(runner_path), str(config_path)],
                               cwd=repo, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            logfile.write(line)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.terminate()
            process.wait()
        raise
if returncode:
    raise RuntimeError(f"Verification stopped with exit code {returncode}. Inspect {log_path}. Run the download cell below to preserve any partial results.")
print("Finished. Download the results below.")


## Download and interpret the results

Download `verification_results.zip` and share it for review. It contains configuration/environment records, test WGA tables for all 30 configurations, fresh model records, code-check output, image-replay mismatches and conditional bootstrap intervals. Large images, backbone checkpoints and raw feature arrays remain in the full notebook output, outside the compact ZIP.

Read the evidence in separate layers:

- **Original result replay:** zero changed reference decisions supports reproduction of saved decisions. Small probability differences alone do not prove identical classifications.
- **Image replay:** inspect all recorded decision changes and preprocessing versions. Do not silently ignore a mismatch.
- **Combiner randomness:** ten fits on one frozen ensemble are ten combiner seeds, not ten independently trained backbones.
- **Dropout-specific effect:** compare dropout MLP with plain MLP as well as uniform averaging. Keep both weight-decay configurations separately reported.
- **Additional source seeds:** these are genuinely new backbone/head runs. Report all planned seeds and retain failures or regressions.

No background-causality or cross-dataset claim follows from these WGA results alone. The notebook does not choose a winning configuration from test results.


In [ ]:
from IPython.display import FileLink, Markdown, display

result_zip = work / "verification_results.zip"
if result_zip.is_file():
    display(FileLink(str(result_zip), result_html_prefix="Download compact verification results: "))
else:
    print("No archive exists yet. Check the setup/worker log and preserve the working directory.")
for report in sorted((work / "results").glob("*/report.md")):
    display(Markdown(f"### {report.parent.name}"))
    display(Markdown(report.read_text()))
print("For backbone resume, preserve the full notebook output, not only the compact ZIP.")
